In [2]:
import json
import os
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [3]:
WINDOW = 223
NUM_FEATURES = 18

In [4]:
current_dir = Path.cwd()
division_data_dir = current_dir.parent.parent / "data" / "division"
window_data_dir = division_data_dir / "data"
test_data_dir = division_data_dir / "test"
division_models_dir = current_dir.parent.parent / "models" / "division"

os.makedirs(test_data_dir, exist_ok=True)
os.makedirs(division_models_dir, exist_ok=True)

In [5]:
targets_df = pd.read_csv(os.path.join(division_data_dir, "targets.csv"))
stream_ranges = pd.read_csv(os.path.join(division_data_dir, "stream_ranges.csv"))

stream_ids = stream_ranges["stream_id"].values

train_streams, temp_streams = train_test_split(
    stream_ids, test_size=0.30, random_state=42, shuffle=True
)

val_streams, test_streams = train_test_split(
    temp_streams, test_size=0.50, random_state=42, shuffle=True
)

print(f"Train streams: {len(train_streams)}")
print(f"Val streams: {len(val_streams)}")
print(f"Test streams: {len(test_streams)}")

def collect_window_ids(stream_subset):
    subset = stream_ranges[stream_ranges["stream_id"].isin(stream_subset)]
    window_ids = []
    for _, row in subset.iterrows():
        first_id = int(row["first_window_id"])
        last_id = int(row["last_window_id"])
        window_ids.extend(range(first_id, last_id + 1))
    return np.array(window_ids, dtype=np.int32)

train_window_ids = collect_window_ids(train_streams)
val_window_ids = collect_window_ids(val_streams)
test_window_ids = collect_window_ids(test_streams)

print(f"Train windows: {len(train_window_ids)}")
print(f"Val windows: {len(val_window_ids)}")
print(f"Test windows: {len(test_window_ids)}")

# Copy test windows to test directory
for file in os.listdir(test_data_dir):
    path = os.path.join(test_data_dir, file)
    if os.path.isfile(path):
        os.remove(path)

for window_id in test_window_ids:
    src = os.path.join(window_data_dir, f"{window_id:06d}.csv")
    dst = os.path.join(test_data_dir, f"{window_id:06d}.csv")
    if os.path.exists(src):
        shutil.copy2(src, dst)

print(f"Copied {len(test_window_ids)} test windows to {test_data_dir}")

Train streams: 350
Val streams: 75
Test streams: 75
Train windows: 11958
Val windows: 2748
Test windows: 2623
Copied 2623 test windows to /Users/dmytrok/Documents/Універ/6 семестр/term6/smart-glove-ml/data/division/test


In [6]:
num_windows = len(targets_df)
X = np.zeros((num_windows, WINDOW, NUM_FEATURES), dtype=np.float32)

for window_id in range(num_windows):
    path = os.path.join(window_data_dir, f"{window_id:06d}.csv")
    df = pd.read_csv(path)
    X[window_id] = df.values.astype(np.float32)

print("Windows loaded")

y_has_start = targets_df["has_start"].values.astype(np.float32)
y_start_norm = targets_df["start_norm"].values.astype(np.float32)
y_has_end = targets_df["has_end"].values.astype(np.float32)
y_end_norm = targets_df["end_norm"].values.astype(np.float32)

# Split
X_train = X[train_window_ids]
X_val = X[val_window_ids]
X_test = X[test_window_ids]

y_train = {
    "has_start": y_has_start[train_window_ids],
    "start_norm": y_start_norm[train_window_ids],
    "has_end": y_has_end[train_window_ids],
    "end_norm": y_end_norm[train_window_ids],
}

y_val = {
    "has_start": y_has_start[val_window_ids],
    "start_norm": y_start_norm[val_window_ids],
    "has_end": y_has_end[val_window_ids],
    "end_norm": y_end_norm[val_window_ids],
}

y_test = {
    "has_start": y_has_start[test_window_ids],
    "start_norm": y_start_norm[test_window_ids],
    "has_end": y_has_end[test_window_ids],
    "end_norm": y_end_norm[test_window_ids],
}

# Scaling
scaler = StandardScaler()

N_train, T, F = X_train.shape
N_val = X_val.shape[0]
N_test = X_test.shape[0]

X_train_2d = X_train.reshape(-1, F)
X_val_2d = X_val.reshape(-1, F)
X_test_2d = X_test.reshape(-1, F)

scaler.fit(X_train_2d)

X_train = scaler.transform(X_train_2d).reshape(N_train, T, F)
X_val = scaler.transform(X_val_2d).reshape(N_val, T, F)
X_test = scaler.transform(X_test_2d).reshape(N_test, T, F)

Windows loaded


In [21]:
joblib.dump(scaler, os.path.join(division_models_dir, "scaler.pkl"))

['/Users/dmytrok/Documents/Універ/6 семестр/term6/smart-glove-ml/models/division/scaler.pkl']

In [7]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, min_delta=1e-4, restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6
)

# Classification models

# Has start model

In [18]:
y_train_start = np.asarray(y_train["has_start"]).astype(np.int32)
y_val_start = np.asarray(y_val["has_start"]).astype(np.int32) 
y_test_start = np.asarray(y_test["has_start"]).astype(np.int32)

print("\nTrain class distribution:") 
print(np.bincount(y_train_start)) 

print("\nValidation class distribution:") 
print(np.bincount(y_val_start)) 

print("\nTest class distribution:") 
print(np.bincount(y_test_start))


# Class Weights

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_start
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print("\nClass weights:")
print(f"Class 0: {class_weights[0]:.4f}")
print(f"Class 1: {class_weights[1]:.4f}")


# Model

has_start_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(T, F)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

has_start_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="roc_auc"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


history = has_start_model.fit(
    X_train,
    y_train_start,
    validation_data=(
        X_val,
        y_val_start
    ),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=1
)


test_results = has_start_model.evaluate(
    X_test,
    y_test_start,
    verbose=1,
    return_dict=True
)

for name, value in test_results.items():
    print(f"{name}: {value:.4f}")


# Receiving probabilities for validation and test sets

val_proba = has_start_model.predict(
    X_val,
    verbose=0
).ravel()

test_proba = has_start_model.predict(
    X_test,
    verbose=0
).ravel()


Train class distribution:
[ 1845 10113]

Validation class distribution:
[ 429 2319]

Test class distribution:
[ 422 2201]

Class weights:
Class 0: 3.2407
Class 1: 0.5912
Epoch 1/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 58ms/step - accuracy: 0.8042 - loss: 0.3514 - pr_auc: 0.9829 - precision: 0.9877 - recall: 0.7782 - roc_auc: 0.9071 - val_accuracy: 0.8668 - val_loss: 0.2962 - val_pr_auc: 0.9917 - val_precision: 0.9900 - val_recall: 0.8508 - val_roc_auc: 0.9530 - learning_rate: 0.0010
Epoch 2/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 60ms/step - accuracy: 0.8734 - loss: 0.2415 - pr_auc: 0.9915 - precision: 0.9908 - recall: 0.8583 - roc_auc: 0.9532 - val_accuracy: 0.9312 - val_loss: 0.1699 - val_pr_auc: 0.9963 - val_precision: 0.9885 - val_recall: 0.9293 - val_roc_auc: 0.9794 - learning_rate: 0.0010
Epoch 3/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - accuracy: 0.9063 - loss: 0.1952 - pr_auc: 0.9945 - precision: 0.9920 - recall: 0.8964 - roc_auc: 0.9695 - val_accuracy: 0.9236 - val_loss: 0.1

In [19]:
# ============================================================
# Threshold selection
#
# We want a balanced classification between class 0 and class 1.
#
# Therefore, instead of optimizing Recall only, we select the
# threshold that gives the highest Macro F1.
#
# Macro F1 = average F1 of class 0 and class 1.
# This gives both classes equal importance.
# ============================================================

thresholds = np.arange(
    0.01,
    1.00,
    0.01
)

threshold_results = []


for threshold in thresholds:

    y_val_pred = (
        val_proba >= threshold
    ).astype(np.int32)

    # F1 for class 0
    f1_class_0 = f1_score(
        y_val_start,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    # F1 for class 1
    f1_class_1 = f1_score(
        y_val_start,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    # Macro F1
    macro_f1 = (
        f1_class_0 + f1_class_1
    ) / 2

    # Standard accuracy
    accuracy = np.mean(
        y_val_start == y_val_pred
    )

    # Precision / Recall for class 0
    precision_0 = precision_score(
        y_val_start,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    recall_0 = recall_score(
        y_val_start,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    # Precision / Recall for class 1
    precision_1 = precision_score(
        y_val_start,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    recall_1 = recall_score(
        y_val_start,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,

        "accuracy": accuracy,

        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_class_0,

        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_class_1,

        "macro_f1": macro_f1
    })


# ============================================================
# Select the best threshold
# ============================================================

best_result = max(
    threshold_results,
    key=lambda x: (
        x["macro_f1"],
        x["accuracy"]
    )
)

best_threshold = best_result["threshold"]


print("\n" + "=" * 60)
print("THRESHOLD SELECTION")
print("=" * 60)

print(
    f"Selected threshold: {best_threshold:.2f}"
)

print(
    f"Validation Accuracy: "
    f"{best_result['accuracy']:.4f}"
)

print(
    f"Validation F1 (class 0): "
    f"{best_result['f1_0']:.4f}"
)

print(
    f"Validation F1 (class 1): "
    f"{best_result['f1_1']:.4f}"
)

print(
    f"Validation Macro F1: "
    f"{best_result['macro_f1']:.4f}"
)

print(
    f"Validation Precision (class 0): "
    f"{best_result['precision_0']:.4f}"
)

print(
    f"Validation Recall (class 0): "
    f"{best_result['recall_0']:.4f}"
)

print(
    f"Validation Precision (class 1): "
    f"{best_result['precision_1']:.4f}"
)

print(
    f"Validation Recall (class 1): "
    f"{best_result['recall_1']:.4f}"
)


# ============================================================
# Final predictions for TEST
#
# IMPORTANT:
# The threshold was selected using validation data.
# We now use the same threshold on the test set.
# ============================================================

y_test_pred = (
    test_proba >= best_threshold
).astype(np.int32)


# ============================================================
# Confusion Matrix
#
# [[TN, FP],
#  [FN, TP]]
# ============================================================

cm = confusion_matrix(
    y_test_start,
    y_test_pred
)

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")


# ============================================================
# TEST METRICS
# ============================================================

test_accuracy = np.mean(
    y_test_start == y_test_pred
)

test_precision_0 = precision_score(
    y_test_start,
    y_test_pred,
    pos_label=0,
    zero_division=0
)

test_recall_0 = recall_score(
    y_test_start,
    y_test_pred,
    pos_label=0,
    zero_division=0
)

test_f1_0 = f1_score(
    y_test_start,
    y_test_pred,
    pos_label=0,
    zero_division=0
)


test_precision_1 = precision_score(
    y_test_start,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_recall_1 = recall_score(
    y_test_start,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_f1_1 = f1_score(
    y_test_start,
    y_test_pred,
    pos_label=1,
    zero_division=0
)


test_macro_f1 = (
    test_f1_0 + test_f1_1
) / 2


test_balanced_accuracy = balanced_accuracy_score(
    y_test_start,
    y_test_pred
)


# ============================================================
# FINAL TEST METRICS
# ============================================================

print("\n" + "=" * 60)
print("FINAL TEST METRICS")
print("=" * 60)

print(
    f"Threshold:             {best_threshold:.2f}"
)

print(
    f"Accuracy:              {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy:     {test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:              {test_macro_f1:.4f}"
)

print("\nClass 0 (no_start):")

print(
    f"  Precision:           {test_precision_0:.4f}"
)

print(
    f"  Recall:              {test_recall_0:.4f}"
)

print(
    f"  F1:                  {test_f1_0:.4f}"
)

print("\nClass 1 (start):")

print(
    f"  Precision:           {test_precision_1:.4f}"
)

print(
    f"  Recall:              {test_recall_1:.4f}"
)

print(
    f"  F1:                  {test_f1_1:.4f}"
)


# ============================================================
# ROC-AUC
# ============================================================

test_roc_auc = roc_auc_score(
    y_test_start,
    test_proba
)

print(
    f"\nROC-AUC:               {test_roc_auc:.4f}"
)


# ============================================================
# PR-AUC
#
# Class 1 (start) is considered positive.
# ============================================================

test_pr_auc = average_precision_score(
    y_test_start,
    test_proba
)

print(
    f"PR-AUC:                {test_pr_auc:.4f}"
)


# ============================================================
# Classification Report
# ============================================================

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test_start,
        y_test_pred,

        target_names=[
            "no_start",
            "start"
        ],

        digits=4,

        zero_division=0
    )
)


# ============================================================
# Threshold comparison:
# default 0.5 vs optimized threshold
# ============================================================

y_test_pred_default = (
    test_proba >= 0.5
).astype(np.int32)


default_accuracy = np.mean(
    y_test_start == y_test_pred_default
)

default_macro_f1 = (
    f1_score(
        y_test_start,
        y_test_pred_default,
        pos_label=0,
        zero_division=0
    )
    +
    f1_score(
        y_test_start,
        y_test_pred_default,
        pos_label=1,
        zero_division=0
    )
) / 2


default_precision_0 = precision_score(
    y_test_start,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)

default_recall_0 = recall_score(
    y_test_start,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)

default_f1_0 = f1_score(
    y_test_start,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)


default_precision_1 = precision_score(
    y_test_start,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)

default_recall_1 = recall_score(
    y_test_start,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)

default_f1_1 = f1_score(
    y_test_start,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)


print("\n" + "=" * 60)
print("THRESHOLD COMPARISON")
print("=" * 60)

print("\nDefault threshold = 0.50")

print(
    f"Accuracy:       {default_accuracy:.4f}"
)

print(
    f"Macro F1:       {default_macro_f1:.4f}"
)

print(
    f"Class 0 F1:     {default_f1_0:.4f}"
)

print(
    f"Class 1 F1:     {default_f1_1:.4f}"
)

print(
    f"Class 0 Recall: {default_recall_0:.4f}"
)

print(
    f"Class 1 Recall: {default_recall_1:.4f}"
)


print(
    f"\nOptimized threshold = "
    f"{best_threshold:.2f}"
)

print(
    f"Accuracy:       {test_accuracy:.4f}"
)

print(
    f"Macro F1:       {test_macro_f1:.4f}"
)

print(
    f"Class 0 F1:     {test_f1_0:.4f}"
)

print(
    f"Class 1 F1:     {test_f1_1:.4f}"
)

print(
    f"Class 0 Recall: {test_recall_0:.4f}"
)

print(
    f"Class 1 Recall: {test_recall_1:.4f}"
)


# ============================================================
# Final summary
# ============================================================

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(
    f"Best threshold:       {best_threshold:.2f}"
)

print(
    f"Accuracy:              {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy:     {test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:              {test_macro_f1:.4f}"
)

print(
    f"F1 class 0:            {test_f1_0:.4f}"
)

print(
    f"F1 class 1:            {test_f1_1:.4f}"
)

print(
    f"ROC-AUC:               {test_roc_auc:.4f}"
)

print(
    f"PR-AUC:                {test_pr_auc:.4f}"
)


THRESHOLD SELECTION
Selected threshold: 0.43
Validation Accuracy: 0.9825
Validation F1 (class 0): 0.9457
Validation F1 (class 1): 0.9896
Validation Macro F1: 0.9676
Validation Precision (class 0): 0.9187
Validation Recall (class 0): 0.9744
Validation Precision (class 1): 0.9952
Validation Recall (class 1): 0.9840

CONFUSION MATRIX
[[ 410   12]
 [  36 2165]]

TN = 410
FP = 12
FN = 36
TP = 2165

FINAL TEST METRICS
Threshold:             0.43
Accuracy:              0.9817
Balanced Accuracy:     0.9776
Macro F1:              0.9669

Class 0 (no_start):
  Precision:           0.9193
  Recall:              0.9716
  F1:                  0.9447

Class 1 (start):
  Precision:           0.9945
  Recall:              0.9836
  F1:                  0.9890

ROC-AUC:               0.9972
PR-AUC:                0.9995

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    no_start     0.9193    0.9716    0.9447       422
       start     0.9945    0.9836    0.9890      2201

In [20]:
has_start_model.save(
    os.path.join(division_models_dir, "has_start.keras")
)

data = {"has_start": best_threshold}

with open(
    os.path.join(division_models_dir, "thresholds.json"),
    "r"
) as f:
    thresholds_data = json.load(f)
    
thresholds_data.update(data)

with open(
    os.path.join(division_models_dir, "thresholds.json"),
    "w"
) as f:
    json.dump(thresholds_data, f, indent=4)

# Has end model

In [15]:
y_train_end = np.asarray(y_train["has_end"]).astype(np.int32)
y_val_end = np.asarray(y_val["has_end"]).astype(np.int32) 
y_test_end = np.asarray(y_test["has_end"]).astype(np.int32)

print("\nTrain class distribution:") 
print(np.bincount(y_train_end)) 

print("\nValidation class distribution:") 
print(np.bincount(y_val_end)) 

print("\nTest class distribution:") 
print(np.bincount(y_test_end))


# Class Weights

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_end
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print("\nClass weights:")
print(f"Class 0: {class_weights[0]:.4f}")
print(f"Class 1: {class_weights[1]:.4f}")


# Model

has_end_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(T, F)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

has_end_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="roc_auc"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


history = has_end_model.fit(
    X_train,
    y_train_end,
    validation_data=(
        X_val,
        y_val_end
    ),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=1
)


test_results = has_end_model.evaluate(
    X_test,
    y_test_end,
    verbose=1,
    return_dict=True
)

for name, value in test_results.items():
    print(f"{name}: {value:.4f}")


# Receiving probabilities for validation and test sets

val_proba = has_end_model.predict(
    X_val,
    verbose=0
).ravel()

test_proba = has_end_model.predict(
    X_test,
    verbose=0
).ravel()


Train class distribution:
[2328 9630]

Validation class distribution:
[ 537 2211]

Test class distribution:
[ 536 2087]

Class weights:
Class 0: 2.5683
Class 1: 0.6209
Epoch 1/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 24s 60ms/step - accuracy: 0.7939 - loss: 0.4508 - pr_auc: 0.9625 - precision: 0.9475 - recall: 0.7876 - roc_auc: 0.8712 - val_accuracy: 0.8639 - val_loss: 0.3153 - val_pr_auc: 0.9797 - val_precision: 0.9581 - val_recall: 0.8688 - val_roc_auc: 0.9269 - learning_rate: 0.0010
Epoch 2/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 58ms/step - accuracy: 0.8511 - loss: 0.3261 - pr_auc: 0.9827 - precision: 0.9672 - recall: 0.8438 - roc_auc: 0.9320 - val_accuracy: 0.8566 - val_loss: 0.3146 - val_pr_auc: 0.9844 - val_precision: 0.9671 - val_recall: 0.8507 - val_roc_auc: 0.9404 - learning_rate: 0.0010
Epoch 3/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 58ms/step - accuracy: 0.8625 - loss: 0.3046 - pr_auc: 0.9847 - precision: 0.9709 - recall: 0.8549 - roc_auc: 0.9420 - val_accuracy: 0.8817 - val_loss: 0.274

In [16]:
# ============================================================
# Threshold selection
#
# We want a balanced classification between class 0 and class 1.
#
# Therefore, instead of optimizing Recall only, we select the
# threshold that gives the highest Macro F1.
#
# Macro F1 = average F1 of class 0 and class 1.
# This gives both classes equal importance.
# ============================================================

thresholds = np.arange(
    0.01,
    1.00,
    0.01
)

threshold_results = []


for threshold in thresholds:

    y_val_pred = (
        val_proba >= threshold
    ).astype(np.int32)

    # F1 for class 0
    f1_class_0 = f1_score(
        y_val_end,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    # F1 for class 1
    f1_class_1 = f1_score(
        y_val_end,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    # Macro F1
    macro_f1 = (
        f1_class_0 + f1_class_1
    ) / 2

    # Standard accuracy
    accuracy = np.mean(
        y_val_end == y_val_pred
    )

    # Precision / Recall for class 0
    precision_0 = precision_score(
        y_val_end,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    recall_0 = recall_score(
        y_val_end,
        y_val_pred,
        pos_label=0,
        zero_division=0
    )

    # Precision / Recall for class 1
    precision_1 = precision_score(
        y_val_end,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    recall_1 = recall_score(
        y_val_end,
        y_val_pred,
        pos_label=1,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,

        "accuracy": accuracy,

        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_class_0,

        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_class_1,

        "macro_f1": macro_f1
    })


# ============================================================
# Select the best threshold
# ============================================================

best_result = max(
    threshold_results,
    key=lambda x: (
        x["macro_f1"],
        x["accuracy"]
    )
)

best_threshold = best_result["threshold"]


print("\n" + "=" * 60)
print("THRESHOLD SELECTION")
print("=" * 60)

print(
    f"Selected threshold: {best_threshold:.2f}"
)

print(
    f"Validation Accuracy: "
    f"{best_result['accuracy']:.4f}"
)

print(
    f"Validation F1 (class 0): "
    f"{best_result['f1_0']:.4f}"
)

print(
    f"Validation F1 (class 1): "
    f"{best_result['f1_1']:.4f}"
)

print(
    f"Validation Macro F1: "
    f"{best_result['macro_f1']:.4f}"
)

print(
    f"Validation Precision (class 0): "
    f"{best_result['precision_0']:.4f}"
)

print(
    f"Validation Recall (class 0): "
    f"{best_result['recall_0']:.4f}"
)

print(
    f"Validation Precision (class 1): "
    f"{best_result['precision_1']:.4f}"
)

print(
    f"Validation Recall (class 1): "
    f"{best_result['recall_1']:.4f}"
)


# ============================================================
# Final predictions for TEST
#
# IMPORTANT:
# The threshold was selected using validation data.
# We now use the same threshold on the test set.
# ============================================================

y_test_pred = (
    test_proba >= best_threshold
).astype(np.int32)


# ============================================================
# Confusion Matrix
#
# [[TN, FP],
#  [FN, TP]]
# ============================================================

cm = confusion_matrix(
    y_test_end,
    y_test_pred
)

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")


# ============================================================
# TEST METRICS
# ============================================================

test_accuracy = np.mean(
    y_test_end == y_test_pred
)

test_precision_0 = precision_score(
    y_test_end,
    y_test_pred,
    pos_label=0,
    zero_division=0
)

test_recall_0 = recall_score(
    y_test_end,
    y_test_pred,
    pos_label=0,
    zero_division=0
)

test_f1_0 = f1_score(
    y_test_end,
    y_test_pred,
    pos_label=0,
    zero_division=0
)


test_precision_1 = precision_score(
    y_test_end,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_recall_1 = recall_score(
    y_test_end,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_f1_1 = f1_score(
    y_test_end,
    y_test_pred,
    pos_label=1,
    zero_division=0
)


test_macro_f1 = (
    test_f1_0 + test_f1_1
) / 2


test_balanced_accuracy = balanced_accuracy_score(
    y_test_end,
    y_test_pred
)


# ============================================================
# FINAL TEST METRICS
# ============================================================

print("\n" + "=" * 60)
print("FINAL TEST METRICS")
print("=" * 60)

print(
    f"Threshold:             {best_threshold:.2f}"
)

print(
    f"Accuracy:              {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy:     {test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:              {test_macro_f1:.4f}"
)

print("\nClass 0 (no_end):")

print(
    f"  Precision:           {test_precision_0:.4f}"
)

print(
    f"  Recall:              {test_recall_0:.4f}"
)

print(
    f"  F1:                  {test_f1_0:.4f}"
)

print("\nClass 1 (end):")

print(
    f"  Precision:           {test_precision_1:.4f}"
)

print(
    f"  Recall:              {test_recall_1:.4f}"
)

print(
    f"  F1:                  {test_f1_1:.4f}"
)


# ============================================================
# ROC-AUC
# ============================================================

test_roc_auc = roc_auc_score(
    y_test_end,
    test_proba
)

print(
    f"\nROC-AUC:               {test_roc_auc:.4f}"
)


# ============================================================
# PR-AUC
#
# Class 1 (end) is considered positive.
# ============================================================

test_pr_auc = average_precision_score(
    y_test_end,
    test_proba
)

print(
    f"PR-AUC:                {test_pr_auc:.4f}"
)


# ============================================================
# Classification Report
# ============================================================

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test_end,
        y_test_pred,

        target_names=[
            "no_end",
            "end"
        ],

        digits=4,

        zero_division=0
    )
)


# ============================================================
# Threshold comparison:
# default 0.5 vs optimized threshold
# ============================================================

y_test_pred_default = (
    test_proba >= 0.5
).astype(np.int32)


default_accuracy = np.mean(
    y_test_end == y_test_pred_default
)

default_macro_f1 = (
    f1_score(
        y_test_end,
        y_test_pred_default,
        pos_label=0,
        zero_division=0
    )
    +
    f1_score(
        y_test_end,
        y_test_pred_default,
        pos_label=1,
        zero_division=0
    )
) / 2


default_precision_0 = precision_score(
    y_test_end,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)

default_recall_0 = recall_score(
    y_test_end,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)

default_f1_0 = f1_score(
    y_test_end,
    y_test_pred_default,
    pos_label=0,
    zero_division=0
)


default_precision_1 = precision_score(
    y_test_end,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)

default_recall_1 = recall_score(
    y_test_end,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)

default_f1_1 = f1_score(
    y_test_end,
    y_test_pred_default,
    pos_label=1,
    zero_division=0
)


print("\n" + "=" * 60)
print("THRESHOLD COMPARISON")
print("=" * 60)

print("\nDefault threshold = 0.50")

print(
    f"Accuracy:       {default_accuracy:.4f}"
)

print(
    f"Macro F1:       {default_macro_f1:.4f}"
)

print(
    f"Class 0 F1:     {default_f1_0:.4f}"
)

print(
    f"Class 1 F1:     {default_f1_1:.4f}"
)

print(
    f"Class 0 Recall: {default_recall_0:.4f}"
)

print(
    f"Class 1 Recall: {default_recall_1:.4f}"
)


print(
    f"\nOptimized threshold = "
    f"{best_threshold:.2f}"
)

print(
    f"Accuracy:       {test_accuracy:.4f}"
)

print(
    f"Macro F1:       {test_macro_f1:.4f}"
)

print(
    f"Class 0 F1:     {test_f1_0:.4f}"
)

print(
    f"Class 1 F1:     {test_f1_1:.4f}"
)

print(
    f"Class 0 Recall: {test_recall_0:.4f}"
)

print(
    f"Class 1 Recall: {test_recall_1:.4f}"
)


# ============================================================
# Final summary
# ============================================================

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(
    f"Best threshold:       {best_threshold:.2f}"
)

print(
    f"Accuracy:              {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy:     {test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:              {test_macro_f1:.4f}"
)

print(
    f"F1 class 0:            {test_f1_0:.4f}"
)

print(
    f"F1 class 1:            {test_f1_1:.4f}"
)

print(
    f"ROC-AUC:               {test_roc_auc:.4f}"
)

print(
    f"PR-AUC:                {test_pr_auc:.4f}"
)


THRESHOLD SELECTION
Selected threshold: 0.26
Validation Accuracy: 0.9283
Validation F1 (class 0): 0.8178
Validation F1 (class 1): 0.9554
Validation Macro F1: 0.8866
Validation Precision (class 0): 0.8125
Validation Recall (class 0): 0.8231
Validation Precision (class 1): 0.9569
Validation Recall (class 1): 0.9539

CONFUSION MATRIX
[[ 442   94]
 [ 119 1968]]

TN = 442
FP = 94
FN = 119
TP = 1968

FINAL TEST METRICS
Threshold:             0.26
Accuracy:              0.9188
Balanced Accuracy:     0.8838
Macro F1:              0.8772

Class 0 (no_end):
  Precision:           0.7879
  Recall:              0.8246
  F1:                  0.8058

Class 1 (end):
  Precision:           0.9544
  Recall:              0.9430
  F1:                  0.9487

ROC-AUC:               0.9719
PR-AUC:                0.9928

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      no_end     0.7879    0.8246    0.8058       536
         end     0.9544    0.9430    0.9487      2087

 

In [17]:
has_end_model.save(
    os.path.join(division_models_dir, "has_end.keras")
)

data = {"has_end": best_threshold}

with open(
    os.path.join(division_models_dir, "thresholds.json"),
    "r"
) as f:
    thresholds_data = json.load(f)
    
thresholds_data.update(data)

with open(
    os.path.join(division_models_dir, "thresholds.json"),
    "w"
) as f:
    json.dump(thresholds_data, f, indent=4)

# Regression model

# Start norm model

In [33]:
start_norm_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(T, F)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

start_norm_model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mae"],
)

history = start_norm_model.fit(
    X_train,
    y_train["start_norm"],
    validation_data=(X_val, y_val["start_norm"]),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

test_loss, test_mae = start_norm_model.evaluate(
    X_test, y_test["start_norm"], verbose=1
)

print(f"Test MAE: {test_mae}")
print(f"Absolute MAE: {test_mae * WINDOW:.4f} samples")

Epoch 1/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 59ms/step - loss: 0.2232 - mae: 0.3065 - val_loss: 0.1980 - val_mae: 0.2697 - learning_rate: 0.0010
Epoch 2/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.1970 - mae: 0.2656 - val_loss: 0.1904 - val_mae: 0.2483 - learning_rate: 0.0010
Epoch 3/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.1912 - mae: 0.2552 - val_loss: 0.1934 - val_mae: 0.2587 - learning_rate: 0.0010
Epoch 4/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.1874 - mae: 0.2475 - val_loss: 0.1902 - val_mae: 0.2435 - learning_rate: 0.0010
Epoch 5/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.1853 - mae: 0.2408 - val_loss: 0.1842 - val_mae: 0.2370 - learning_rate: 0.0010
Epoch 6/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.1817 - mae: 0.2340 - val_loss: 0.1800 - val_mae: 0.2266 - learning_rate: 0.0010
Epoch 7/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 54s 144ms/step - loss: 0.1802 - mae: 0.2319 - val_loss: 0.1779 - val_mae: 0.2255 - learni

In [34]:
start_norm_model.save(
    os.path.join(division_models_dir, "start_norm.keras")
)

# End norm model

In [8]:
end_norm_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(T, F)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

end_norm_model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mae"],
)

history = end_norm_model.fit(
    X_train,
    y_train["end_norm"],
    validation_data=(X_val, y_val["end_norm"]),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

test_loss, test_mae = end_norm_model.evaluate(
    X_test, y_test["end_norm"], verbose=1
)

print(f"Test MAE: {test_mae}")
print(f"Absolute MAE: {test_mae * WINDOW:.4f} samples")

Epoch 1/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 23s 59ms/step - loss: 0.3057 - mae: 0.3808 - val_loss: 0.2603 - val_mae: 0.3380 - learning_rate: 0.0010
Epoch 2/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2633 - mae: 0.3369 - val_loss: 0.2576 - val_mae: 0.3313 - learning_rate: 0.0010
Epoch 3/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2538 - mae: 0.3210 - val_loss: 0.2595 - val_mae: 0.3211 - learning_rate: 0.0010
Epoch 4/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2492 - mae: 0.3145 - val_loss: 0.2613 - val_mae: 0.3122 - learning_rate: 0.0010
Epoch 5/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2501 - mae: 0.3149 - val_loss: 0.2496 - val_mae: 0.3077 - learning_rate: 0.0010
Epoch 6/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2478 - mae: 0.3126 - val_loss: 0.2482 - val_mae: 0.3141 - learning_rate: 0.0010
Epoch 7/100
374/374 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 0.2467 - mae: 0.3130 - val_loss: 0.2455 - val_mae: 0.3081 - learnin

In [9]:
end_norm_model.save(
    os.path.join(division_models_dir, "end_norm.keras")
)